#### Task 1: Simple vector embedding generation

**Objective:**
Generate vector embeddings from text data.

**Task Description:**

- load embedding model into ollama:
    1. Attach ollama container shell
    2. Run command in container `ollama pull pull bge-m3` or send an api call `requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False}` via the requests library to download embedding model.
- embed simple text queries

How to select the right embedding model: [MTEB - Massive Text Embedding Benchmark](https://huggingface.co/blog/mteb)

**Useful links:**

- [Ollama REST API](https://www.postman.com/postman-student-programs/ollama-api/documentation/suc47x8/ollama-rest-api)
- [Langchain Ollama Embeddings](https://python.langchain.com/docs/integrations/text_embedding/ollama/)
- [Langchain Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/)


In [15]:
import requests

In [16]:
# Download embedding model into ollama container
requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False})

<Response [200]>

In [17]:
# List all available models in the ollama container
response = requests.get("http://localhost:11434/api/tags")
response.json()

{'models': [{'name': 'bge-m3:latest',
   'model': 'bge-m3:latest',
   'modified_at': '2026-11-04T11:49:38.3106125+01:00',
   'size': 1157672605,
   'digest': '7907646426070047a77226ac3e684fbbe8410524f7b4a74d02837e43f2146bab',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'bert',
    'families': ['bert'],
    'parameter_size': '566.70M',
    'quantization_level': 'F16',
    'context_length': 8192,
    'embedding_length': 1024,
    'runner': 'ggml'},
   'capabilities': ['embedding']},
  {'name': 'llama3.2:1b',
   'model': 'llama3.2:1b',
   'modified_at': '2026-10-15T12:22:11.5113462+02:00',
   'size': 1321098329,
   'digest': 'baf6a787fdffd633537aa2eb51cfd54cb93ff08e28040095462bb63daf552878',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'llama',
    'families': ['llama'],
    'parameter_size': '1.2B',
    'quantization_level': 'Q8_0',
    'context_length': 131072,
    'embedding_length': 2048,
    'runner': 'ggml'},
   'capabilities': ['

In [18]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="bge-m3")

In [19]:
text = "This is a test document."

# ADD HERE YOUR CODE
# Perform vector search
query_vector = embedding_model.embed_query(text)

print(f"Embedding vector length: {len(query_vector)}")
print(query_vector[:10])

Embedding vector length: 1024
[-0.049998682, 0.021091571, -0.057094406, -0.005516551, -0.006297905, -0.02841459, 0.045169096, 0.04271729, -0.00022960188, 0.013049353]


#### Task 2: Generate embedding vectors with custom dataset

**Objective:**
Load custom dataset, preprocess it and generate vector embeddings.

**Task Description:**

- load pdf document "AI_Book.pdf" via langchain document loader: ``PyPDFLoader``
- use RecursiveCharacterTextSplitter to split documents into chunks
- generate embeddings for single documents

**RecursiveCharacterTextSplitter:**

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

**Useful links:**

- [Langchain PyPDFLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader/)
- [Langchain RecursiveCharacterTextSplitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)


In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

pdf_doc = "./AI_Book.pdf"

# Create pdf data loader
# ADD HERE YOUR CODE
loader = PyPDFLoader(pdf_doc)

# Load and split documents in chunks
# ADD HERE YOUR CODE
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
pages_chunked = loader.load_and_split(text_splitter)

print(pages_chunked[0])

# Function to clean text by removing invalid unicode characters, including surrogate pairs
def clean_text(text):
    # Remove surrogate pairs
    text = re.sub(r'[\ud800-\udfff]', '', text)
    # Optionally remove non-ASCII characters (depends on your use case)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

pages_chunked_cleaned = [
    Document(page_content=clean_text(doc.page_content), metadata=doc.metadata)
    for doc in pages_chunked
]

print(pages_chunked_cleaned[0])


page_content='Aurélien Géron
Hands-on Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2019-05-07T15:51:31+00:00', 'moddate': '2019-05-07T15:51:31+00:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': './AI_Book.pdf', 'total_pages': 510, 'page': 2, 'page_label': '1'}
page_content='Aurlien Gron
Hands-on Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creat

In [21]:
print(pages_chunked_cleaned[1])

page_content='978-1-492-03264-9
[LSI]
Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow
by Aurlien Gron
Copyright  2019 Aurlien Gron. All rights reserved.
Printed in the United States of America.
Published by OReilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
OReilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles (http://oreilly.com). For more information, contact our corporate/institutional
sales department: 800-998-9938 or corporate@oreilly.com.
Editor: Nicole Tache
Interior Designer: David Futato
Cover Designer: Karen Montgomery
Illustrator: Rebecca Demarest
June 2019:  Second Edition
Revision History for the Early Release
2018-11-05: First Release
2019-01-24: Second Release
2019-03-07: Third Release
2019-03-29: Fourth Release
2019-04-22: Fifth Release
See http://oreilly.com/catalog/errata.csp?isbn=9781492032649 for release details.' metadata={'producer': 'Antenna

In [22]:
print(f"Number of text chunks: {len(pages_chunked_cleaned)}")

Number of text chunks: 1267


#### Task 3: Store vector embeddings from pdf document to ChromaDB vector database.

**Objective:**

Store vector embeddings into ChromaDB to store knowledge.

**Task Description:**

- create chromadb client
- create chromadb collection
- create langchain chroma db client
- store text document chunks and vector embeddings to vector databases

**Useful links:**

- [Langchain How To](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#initialization-from-client)

In [23]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

collection_name = "ai_model_book"

# ADD HERE YOUR CODE
# Create a collection
collection = client.get_or_create_collection(name=collection_name)

# ADD HERE YOUR CODE
# Create chromadb
vector_db_from_client = Chroma(
    client=client, 
    collection_name=collection_name, 
    embedding_function=embedding_model,
)

In [25]:
from uuid import uuid4
import time

uuids = [str(uuid4()) for _ in range(len(pages_chunked_cleaned))]

# Wir verarbeiten die Dokumente in kleinen 10er-Blöcken (Batches)
batch_size = 10 
total_chunks = len(pages_chunked_cleaned)

print(f"Starte Upload von {total_chunks} Chunks in Batches von {batch_size}...")

for i in range(0, total_chunks, batch_size):
    batch_docs = pages_chunked_cleaned[i : i + batch_size]
    batch_ids = uuids[i : i + batch_size]
    
    print(f"Verarbeite Chunk {i} bis {i + len(batch_docs)} von {total_chunks}...")
    
    # Füge nur den kleinen Batch hinzu
    vector_db_from_client.add_documents(documents=batch_docs, ids=batch_ids)
    
    # Kurze Atempause für den Arbeitsspeicher
    time.sleep(1) 

print("Upload erfolgreich abgeschlossen!")


Starte Upload von 1267 Chunks in Batches von 10...
Verarbeite Chunk 0 bis 10 von 1267...
Verarbeite Chunk 10 bis 20 von 1267...
Verarbeite Chunk 20 bis 30 von 1267...
Verarbeite Chunk 30 bis 40 von 1267...
Verarbeite Chunk 40 bis 50 von 1267...
Verarbeite Chunk 50 bis 60 von 1267...
Verarbeite Chunk 60 bis 70 von 1267...
Verarbeite Chunk 70 bis 80 von 1267...
Verarbeite Chunk 80 bis 90 von 1267...
Verarbeite Chunk 90 bis 100 von 1267...
Verarbeite Chunk 100 bis 110 von 1267...
Verarbeite Chunk 110 bis 120 von 1267...
Verarbeite Chunk 120 bis 130 von 1267...
Verarbeite Chunk 130 bis 140 von 1267...
Verarbeite Chunk 140 bis 150 von 1267...
Verarbeite Chunk 150 bis 160 von 1267...
Verarbeite Chunk 160 bis 170 von 1267...
Verarbeite Chunk 170 bis 180 von 1267...
Verarbeite Chunk 180 bis 190 von 1267...
Verarbeite Chunk 190 bis 200 von 1267...
Verarbeite Chunk 200 bis 210 von 1267...
Verarbeite Chunk 210 bis 220 von 1267...
Verarbeite Chunk 220 bis 230 von 1267...
Verarbeite Chunk 230 bis 2

In [26]:
client.count_collections()

1

In [89]:
# client.delete_collection("ai_model_book")

In [27]:
# Get and check the content of the collection
collection.get(include=["documents", "metadatas", "embeddings"])

{'ids': ['ab01730c-cb9b-497e-ae08-db15df82c423',
  '12bdfeb5-7673-49e2-9e8f-03018afe9065',
  '5afaacaa-c349-448c-b132-9dc656ed731a',
  '8c929e16-9970-4f6f-ac75-6291d885f4b3',
  '32e5089d-bd10-4494-aa1f-5e97296ca536',
  'a2c75e92-e811-413c-b758-b725c1f1e65b',
  'c5d3b63c-9242-458f-a166-1c89acb244c4',
  '9569b09b-49a1-42b7-8497-149ef8a60458',
  '4f3e9fd3-c2e5-4317-a1b1-7f0158f6dca0',
  '844e4178-eea6-482d-b194-6d6ba8aedad9',
  '1dea54f2-3117-4bb8-a200-e1d9456443d7',
  '7b92f2fe-6f03-4c5d-af7f-35ab80fe59dd',
  '964b1682-fcc1-4b9c-8ea8-1f57c8969e83',
  '1b43bb8d-7a0f-47ec-9a22-63e358ca2c9a',
  '1c1efe28-9b3d-4076-8d8e-2827665dc0da',
  'c96aa9a2-8c03-4db1-87cf-4cb158ecdae1',
  '708233fb-8021-47a2-8227-e877805af1c7',
  'a224bbaa-b4e4-4de0-ac6c-b2e76dc0efa0',
  'b18ed605-3cff-49e7-8e7c-9d34d3de5616',
  '0d76c4e3-7447-42c4-b43b-979154a76714',
  'c39c07e0-3ae0-4cad-a017-eca5db2ebdcd',
  'e7c90fdc-ef91-4df4-afd7-41ecac93186c',
  '66288f18-b5b4-4a7f-8fa5-bf8a1db8a026',
  'b1f7d76c-7a6f-48ef-bfd5-

#### Task 4: Access ChromaDB and perform vector search

**Objective:**

Use query to perform vector search against ChromaDB vector database

**Task Description:**

- define query
- run vector search
- print k=3 most similar documents


**Useful links:**

- [Langchain Query ChromaDB](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#search-by-vector)

In [28]:
search_query = "Types of Machine Learning Systems"

results = vector_db_from_client.similarity_search(search_query, k=3)

for res in results:
    print(res.page_content)
    print(res.metadata)
    print("\n")

Types of Machine Learning Systems
There are so many different types of Machine Learning systems that it is useful to
classify them in broad categories based on:
 Whether or not they are trained with human supervision (supervised, unsuper
vised, semisupervised, and Reinforcement Learning)
 Whether or not they can learn incrementally on the fly (online versus batch
learning)
 Whether they work by simply comparing new data points to known data points,
or instead detect patterns in the training data and build a predictive model, much
like scientists do (instance-based versus model-based learning)
These criteria are not exclusive; you can combine them in any way you like. For
example, a state-of-the-art spam filter may learn on the fly using a deep neural net
work model trained using examples of spam and ham; this makes it an online, model-
based, supervised learning system.
Lets look at each of these criteria a bit more closely.
Supervised/Unsupervised Learning
{'trapped': '/False', 'page'